[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_07_research_capstone/ch35_packaging_deployment_reproducibility/01_walkthrough.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 35 Walkthrough: Publishing Reproducible Research — Model Cards, Inference Contracts, Artifact Manifests & TRIPOD+AI Checklists

**Book Reference:** Chapter 35 (Part VII — Research Capstone), Sections 35.1 – 35.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every file write runs strictly inside `tempfile.TemporaryDirectory()` so zero temporary artifacts pollute your workspace.

In [ ]:
# @title Step 0: Click Play (▶) to Load Packages & Train Baseline Pipeline
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
from datetime import datetime, timezone
import hashlib
import json
import math
from pathlib import Path
import platform
import tempfile
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import sharukh_tutor as tutor

mch_raw = tutor.load_data("mch_survey_sample", as_pandas=True)
mch = mch_raw.copy()
mch["residence"] = (
    mch["residence"].astype(str).str.strip().str.title()
)
mch["wealth_quintile"] = (
    mch["wealth_quintile"].astype(str).str.strip().str.title()
)
num_cols = ["maternal_age_years", "anc_visits", "maternal_hb_g_dl"]
cat_cols = ["residence", "wealth_quintile"]
mch[num_cols] = mch[num_cols].replace(999, np.nan)

X = mch[num_cols + cat_cols]
y = mch["child_stunted"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

prep = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore", sparse_output=False
        )),
    ]), cat_cols),
])
pipe = Pipeline([
    ("prep", prep),
    ("model", LogisticRegression(max_iter=500)),
])
pipe.fit(X_train, y_train)
test_probs = pipe.predict_proba(X_test)[:, 1]
test_auc = round(float(roc_auc_score(y_test, test_probs)), 4)
test_brier = round(float(brier_score_loss(y_test, test_probs)), 4)
print(f"Train/Test split: {len(X_train)}/{len(X_test)} | "
      f"Test ROC-AUC: {test_auc:.4f} | Brier: {test_brier:.4f}")


## Section 35.3.1 — Strict Clinical Payload Contract (`validate_clinical_payload`)

In [ ]:
REQUIRED_KEYS = {
    "maternal_age_years",
    "anc_visits",
    "maternal_hb_g_dl",
    "residence",
    "wealth_quintile",
}
VALID_RESIDENCE = {"Rural", "Urban"}
VALID_WEALTH = {"Poorest", "Poorer", "Middle", "Richer", "Richest"}

def _check_num(val, lo, hi, field):
    if (
        isinstance(val, bool)
        or not isinstance(val, (int, float))
        or not math.isfinite(float(val))
    ):
        raise ValueError(f"{field} must be a finite number: {val!r}")
    num = float(val)
    if not (lo <= num <= hi):
        raise ValueError(f"{field} outside [{lo}, {hi}]: {num}")
    return num

def validate_clinical_payload(payload):
    """Validate & normalize a single maternal risk request dict."""
    if not isinstance(payload, dict):
        raise ValueError("Payload must be a dictionary.")
    missing = sorted(REQUIRED_KEYS - set(payload.keys()))
    if missing:
        raise ValueError(f"Missing required keys: {missing}")
    age = _check_num(
        payload["maternal_age_years"], 12, 60, "maternal_age_years"
    )
    anc = _check_num(payload["anc_visits"], 0, 25, "anc_visits")
    hb = _check_num(
        payload["maternal_hb_g_dl"], 2.0, 25.0, "maternal_hb_g_dl"
    )
    res = payload["residence"]
    if not isinstance(res, str):
        raise ValueError("residence must be a string.")
    res_clean = res.strip().title()
    if res_clean not in VALID_RESIDENCE:
        raise ValueError(f"Invalid residence: {res!r}")
    wq = payload["wealth_quintile"]
    if not isinstance(wq, str):
        raise ValueError("wealth_quintile must be a string.")
    wq_clean = wq.strip().title()
    if wq_clean not in VALID_WEALTH:
        raise ValueError(f"Invalid wealth_quintile: {wq!r}")
    return {
        "maternal_age_years": age,
        "anc_visits": anc,
        "maternal_hb_g_dl": hb,
        "residence": res_clean,
        "wealth_quintile": wq_clean,
    }

messy_req = {
    "maternal_age_years": 17,
    "anc_visits": 1,
    "maternal_hb_g_dl": 14.2,
    "residence": "  rural ",
    "wealth_quintile": " poorest ",
}
clean_req = validate_clinical_payload(messy_req)
print("Normalized payload:", clean_req)
print("Caller dict mutated?:", messy_req["residence"] != "  rural ")


## Section 35.3.2 — `ClinicalRiskService` & Production FastAPI Endpoint Pattern

In [ ]:
class ClinicalRiskService:
    """Versioned clinical inference wrapper with risk tiering."""

    def __init__(self, pipeline, threshold=0.35, model_version="v1.0"):
        self.pipeline = pipeline
        self.threshold = float(threshold)
        self.model_version = str(model_version)

    def predict_one(self, payload):
        clean = validate_clinical_payload(payload)
        row = pd.DataFrame([clean])
        prob = float(self.pipeline.predict_proba(row)[0, 1])
        label = int(prob >= self.threshold)
        if prob >= self.threshold:
            tier = "High"
        elif prob >= self.threshold - 0.15:
            tier = "Borderline"
        else:
            tier = "Low"
        return {
            "probability": round(prob, 4),
            "predicted_label": label,
            "risk_tier": tier,
            "threshold": self.threshold,
            "model_version": self.model_version,
        }

service = ClinicalRiskService(
    pipe, threshold=0.35, model_version="v1.0"
)
sample_profiles = [
    messy_req,
    {
        "maternal_age_years": 28,
        "anc_visits": 1,
        "maternal_hb_g_dl": 9.2,
        "residence": "Rural",
        "wealth_quintile": "Poorest",
    },
    {
        "maternal_age_years": 30,
        "anc_visits": 6,
        "maternal_hb_g_dl": 12.8,
        "residence": "Urban",
        "wealth_quintile": "Richest",
    },
]
for idx, prof in enumerate(sample_profiles, start=1):
    print(f"Case {idx}:", service.predict_one(prof))


### FastAPI Deployment Pattern (Reference Blueprint — No Live Server Required)

Because `ClinicalRiskService.predict_one(payload)` owns both schema validation and threshold logic, wrapping it inside a **FastAPI** microservice takes under 15 lines:

```python
# api/app.py  (illustrative production entrypoint)
from fastapi import FastAPI, HTTPException
import joblib

app = FastAPI(title="MCH Stunting Risk API", version="1.0")
svc = ClinicalRiskService(joblib.load("model.joblib"), threshold=0.35)

@app.post("/v1/predict")
def predict_endpoint(payload: dict):
    try:
        return svc.predict_one(payload)
    except ValueError as exc:
        raise HTTPException(status_code=422, detail=str(exc))
```

## Section 35.3.3 — Programmatic Model Card Generation (`generate_model_card_markdown`)

In [ ]:
def generate_model_card_markdown(meta):
    """Render a 5-section clinical Model Card in Markdown."""
    lims = meta["limitations"]
    if isinstance(lims, list):
        lim_text = "\n".join(f"- {item}" for item in lims)
    else:
        lim_text = str(lims)
    return (
        f"# Model Card: {meta['name']}\n\n"
        f"## Model Details\n"
        f"- **Name:** {meta['name']}\n"
        f"- **Version:** {meta['version']}\n\n"
        f"## Intended Use\n"
        f"{meta['intended_use']}\n\n"
        f"## Training Data\n"
        f"{meta['dataset']}\n\n"
        f"## Evaluation Metrics\n"
        f"- **ROC-AUC:** {float(meta['roc_auc']):.4f}\n"
        f"- **Brier Score:** {float(meta['brier_score']):.4f}\n\n"
        f"## Fairness & Limitations\n"
        f"{lim_text}\n"
    )

card_meta = {
    "name": "MCH Child Stunting Triage Classifier",
    "version": "v1.0",
    "intended_use": (
        "Community screening triage for nutritional follow-up."
    ),
    "dataset": "mch_survey_sample (n=500, 75/25 stratified split)",
    "roc_auc": test_auc,
    "brier_score": test_brier,
    "limitations": (
        "Cross-sectional survey; external clinic validation required."
    ),
}
card_md = generate_model_card_markdown(card_meta)
print(card_md)


## Section 35.3.4 — Cryptographic Artifact Manifests & Tamper Detection

In [ ]:
def build_reproducibility_bundle(artifact_path, config_dict, out_dir):
    """Hash artifact with SHA-256 and write config & manifest JSON."""
    artifact = Path(artifact_path)
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    sha256 = hashlib.sha256(artifact.read_bytes()).hexdigest()
    manifest = {
        "artifact_name": artifact.name,
        "artifact_bytes": int(artifact.stat().st_size),
        "artifact_sha256": sha256,
        "python_version": platform.python_version(),
        "created_utc": datetime.now(timezone.utc).strftime(
            "%Y-%m-%dT%H:%M:%SZ"
        ),
    }
    (out / "config.json").write_text(
        json.dumps(config_dict, indent=2), encoding="utf-8"
    )
    (out / "manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )
    return manifest

def verify_reproducibility_bundle(out_dir, artifact_path):
    """Verify SHA-256 digest and config presence for a bundle."""
    out = Path(out_dir)
    manifest = json.loads(
        (out / "manifest.json").read_text(encoding="utf-8")
    )
    actual_sha = hashlib.sha256(
        Path(artifact_path).read_bytes()
    ).hexdigest()
    hash_match = actual_sha == manifest["artifact_sha256"]
    config_present = (out / "config.json").exists()
    return {
        "hash_match": bool(hash_match),
        "config_present": bool(config_present),
        "bundle_valid": bool(hash_match and config_present),
    }

with tempfile.TemporaryDirectory() as tmpdir:
    bundle_dir = Path(tmpdir) / "release_v1"
    bundle_dir.mkdir()
    model_file = bundle_dir / "model.joblib"
    joblib.dump(pipe, model_file)
    cfg = {"seed": 42, "threshold": 0.35, "split": "75/25"}
    man = build_reproducibility_bundle(model_file, cfg, bundle_dir)
    print("Artifact bytes:", man["artifact_bytes"],
          "| SHA-256 prefix:", man["artifact_sha256"][:16])
    print("Untampered check:", verify_reproducibility_bundle(
        bundle_dir, model_file
    ))
    # Simulate 1-byte accidental overwrite / corruption
    model_file.write_bytes(model_file.read_bytes() + b"X")
    print("Tampered check:  ", verify_reproducibility_bundle(
        bundle_dir, model_file
    ))


## Section 35.3.5 — Inference Latency & p95 SLA Benchmarking

In [ ]:
times_ms = []
for _ in range(50):
    t0 = time.perf_counter()
    service.predict_one(messy_req)
    times_ms.append((time.perf_counter() - t0) * 1000.0)

mean_ms = float(np.mean(times_ms))
p95_ms = float(np.percentile(times_ms, 95))
max_ms = float(np.max(times_ms))
tps = 1000.0 / max(mean_ms, 1e-9)
print(f"n=50 | mean={mean_ms:.3f} ms | p95={p95_ms:.3f} ms | "
      f"max={max_ms:.3f} ms | tps={tps:.1f} req/s")


## Section 35.3.6 — TRIPOD+AI 8-Point Reporting Audit (`score_tripod_ai_checklist`)

In [ ]:
TRIPOD_ITEMS = [
    "calibration_reported",
    "code_released",
    "data_source_described",
    "leakage_prevented",
    "limitations_stated",
    "missing_data_handled",
    "seeds_fixed",
    "subgroup_fairness_audited",
]

def score_tripod_ai_checklist(checklist_dict):
    """Audit 8 mandatory TRIPOD+AI items and flag missing items."""
    missing = sorted(
        k for k in TRIPOD_ITEMS
        if not bool(checklist_dict.get(k, False))
    )
    score = len(TRIPOD_ITEMS) - len(missing)
    return {
        "score": int(score),
        "max_score": 8,
        "pct": round(100.0 * score / 8, 1),
        "missing_items": missing,
    }

draft_audit = {
    "calibration_reported": False,
    "code_released": True,
    "data_source_described": True,
    "leakage_prevented": True,
    "limitations_stated": True,
    "missing_data_handled": True,
    "seeds_fixed": False,
}
final_audit = {item: True for item in TRIPOD_ITEMS}
print("Draft manuscript:", score_tripod_ai_checklist(draft_audit))
print("Final release:   ", score_tripod_ai_checklist(final_audit))


## Section 35.3.7 — Break It On Purpose! (Contract & Manifest Error Anatomy)

In [ ]:
# Bug #1: ValueError — impossible hemoglobin 99.0 g/dL rejected
validate_clinical_payload({
    "maternal_age_years": 26,
    "anc_visits": 4,
    "maternal_hb_g_dl": 99.0,
    "residence": "Rural",
    "wealth_quintile": "Middle",
})


In [ ]:
# Bug #2: ValueError — missing required key 'residence'
validate_clinical_payload({
    "maternal_age_years": 26,
    "anc_visits": 4,
    "maternal_hb_g_dl": 11.2,
    "wealth_quintile": "Middle",
})


In [ ]:
# Bug #3: FileNotFoundError — verifying bundle with missing manifest
with tempfile.TemporaryDirectory() as empty_dir:
    dummy = Path(empty_dir) / "model.joblib"
    dummy.write_bytes(b"placeholder")
    verify_reproducibility_bundle(empty_dir, dummy)


## Section 35.4 — Stage 4 Blueprint: End-to-End Reproducible Release Pipeline

In [ ]:
def audit_and_release_clinical_model(raw_df, seed=42):
    """Train, package, hash, verify & audit inside a temp directory."""
    df = raw_df.copy()
    df["residence"] = (
        df["residence"].astype(str).str.strip().str.title()
    )
    df["wealth_quintile"] = (
        df["wealth_quintile"].astype(str).str.strip().str.title()
    )
    n_cols = ["maternal_age_years", "anc_visits", "maternal_hb_g_dl"]
    c_cols = ["residence", "wealth_quintile"]
    df[n_cols] = df[n_cols].replace(999, np.nan)
    X_all, y_all = df[n_cols + c_cols], df["child_stunted"]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_all, y_all, test_size=0.25, stratify=y_all, random_state=seed
    )
    pre = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), n_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore", sparse_output=False
            )),
        ]), c_cols),
    ])
    mdl = Pipeline([
        ("prep", pre),
        ("model", LogisticRegression(max_iter=500)),
    ])
    mdl.fit(X_tr, y_tr)
    pr = mdl.predict_proba(X_te)[:, 1]
    auc_v = round(float(roc_auc_score(y_te, pr)), 4)
    brier_v = round(float(brier_score_loss(y_te, pr)), 4)

    with tempfile.TemporaryDirectory() as tmpdir:
        out = Path(tmpdir) / "mch_release_v1"
        out.mkdir()
        m_path = out / "model.joblib"
        joblib.dump(mdl, m_path)
        meta = {
            "name": "MCH Child Stunting Risk Model",
            "version": "v1.0",
            "intended_use": "Community screening triage.",
            "dataset": "mch_survey_sample (n=500)",
            "roc_auc": auc_v,
            "brier_score": brier_v,
            "limitations": "Requires external prospective audit.",
        }
        (out / "MODEL_CARD.md").write_text(
            generate_model_card_markdown(meta), encoding="utf-8"
        )
        cfg = {"seed": seed, "threshold": 0.35, "test_auc": auc_v}
        build_reproducibility_bundle(m_path, cfg, out)
        ver = verify_reproducibility_bundle(out, m_path)
        files = sorted(p.name for p in out.iterdir() if p.is_file())

    tripod = score_tripod_ai_checklist({k: True for k in TRIPOD_ITEMS})
    return {
        "test_auc": auc_v,
        "brier_score": brier_v,
        "bundle_valid": ver["bundle_valid"],
        "tripod_pct": tripod["pct"],
        "files_created": files,
    }

print(audit_and_release_clinical_model(mch_raw))
